# 02 — BCRD CPI & Family Basket | Automatic Download (raw landing)

Downloads the six BCRD price-index workbooks and lands them, unchanged, in the raw archive
`/Volumes/rd_economic_intelligence/bronze/bcrd_raw/cpi/`.

| Step | Rule |
|---|---|
| Source | Stable URLs under `https://cdn.bancentral.gov.do/documents/estadisticas/precios/documents/`. Core CPI and basket by quintile are published as `.xlsx`; national CPI, CPI by group, CPI by region and basket by region only as legacy Excel `.xls` (read in Bronze with `xlrd`). |
| Validate | Each file must open as a workbook and contain `Período` in the first 10 rows of its first sheet; otherwise the task fails and nothing is written. |
| Land | Shared helper `src/shared/acquisition.py` (identical content never landed twice; republished names kept as `<name>__<date>`). |
| Signal | Task value `downloaded` = true when at least one new file landed. |

**Status:** Development.

In [0]:
%pip install openpyxl==3.1.5 xlrd==2.0.1 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import sys, io, json, importlib, itertools
import openpyxl, xlrd

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import acquisition as ACQ
importlib.reload(ACQ)

LANDING = "/Volumes/rd_economic_intelligence/bronze/bcrd_raw/cpi/"
BASE = "https://cdn.bancentral.gov.do/documents/estadisticas/precios/documents/"
FILES = ["ipc_base_2019-2020.xls", "ipc_grupos_base_2019-2020.xls", "ipc_subyacente_base_2019-2020.xlsx",
         "ipc_regiones_base_2019-2020.xls", "Costo_Canasta_quintiles_base_2019-2020.xlsx", "Costo_Canasta_regiones_base_2019-2020.xls"]

def first_rows_text(name, data):
    if name.lower().endswith(".xls"):
        sh = xlrd.open_workbook(file_contents=data).sheet_by_index(0)
        return " ".join(str(sh.cell_value(r, c)) for r in range(min(10, sh.nrows)) for c in range(sh.ncols))
    ws = openpyxl.load_workbook(io.BytesIO(data), read_only=True, data_only=True).worksheets[0]
    return " ".join(str(c) for r in itertools.islice(ws.iter_rows(values_only=True), 0, 10) for c in r if c is not None)

results = {}
for name in FILES:
    data = ACQ.http_get(BASE + name)
    if "Período" not in first_rows_text(name, data):
        raise ValueError(f"{name}: structural contract failed ('Período' not found in the first 10 rows)")
    results[name] = ACQ.land(data, name, LANDING)
downloaded = any(r["status"] == "DOWNLOADED" for r in results.values())
dbutils.jobs.taskValues.set(key="downloaded", value=downloaded)
dbutils.notebook.exit(json.dumps({"status": "DOWNLOADED" if downloaded else "SKIPPED", "files": results}, ensure_ascii=False))